In [1]:
import torch
import torch.nn.functional as F
import requests
import time
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler

import gc
import os
import json
import uuid
import platform
from zipfile import ZipFile

import klastroknowledge as km


In [2]:
# Column names for the Adult dataset
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
		   'marital-status', 'occupation', 'relationship', 'race', 'sex',
		   'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

# Load from UCI ML repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)

print(f"Shape: {df.shape}")
print(df.head())

Shape: (32561, 15)
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0        

In [3]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [4]:
numerical = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [5]:
def restrict_to_iqr_range(data, columns, lower_q=0.25, upper_q=0.75):
	for col in columns:
		original_dtype = data[col].dtype
		# 10% - 90% quantile
		lower = data[col].quantile(lower_q)
		upper = data[col].quantile(upper_q)
		
		# IQR range
		iqr = upper - lower
		lower_bound = lower - 1.5 * iqr  # lower limit
		upper_bound = upper + 1.5 * iqr  # upper limit
		
		# data with an IQR range\
		data[col] = data[col].clip(lower=lower_bound, upper=upper_bound)
		
		if original_dtype == 'float64':
			data[col] = data[col].astype('float32')
		elif original_dtype == 'int64':
			data[col] = data[col].astype('int64')  # Redundant but keeps the intention clear
	return data

In [6]:
df = restrict_to_iqr_range(df.copy(), numerical, lower_q=0.25, upper_q=0.75)

for col in numerical:
	df[col] = pd.to_numeric(df[col], errors='coerce')

df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,32,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,32,United-States,<=50K


In [7]:
scaler = StandardScaler()

# Fit and transform the numerical columns
df[numerical] = scaler.fit_transform(df[numerical])
df


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.032782,State-gov,-1.149114,Bachelors,1.156144,Never-married,Adm-clerical,Not-in-family,White,Male,0.0,0.0,-0.171298,United-States,<=50K
1,0.844236,Self-emp-not-inc,-1.088196,Bachelors,1.156144,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,-1.459102,United-States,<=50K
2,-0.040986,Private,0.302927,HS-grad,-0.441802,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,-0.171298,United-States,<=50K
3,1.065541,Private,0.503446,11th,-1.240775,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,-0.171298,United-States,<=50K
4,-0.778671,Private,1.593428,Bachelors,1.156144,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,-0.171298,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,-0.852440,Private,0.740821,Assoc-acdm,0.756658,Married-civ-spouse,Tech-support,Wife,White,Female,0.0,0.0,-0.493249,United-States,<=50K
32557,0.106551,Private,-0.341172,HS-grad,-0.441802,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0.0,0.0,-0.171298,United-States,>50K
32558,1.434384,Private,-0.367074,HS-grad,-0.441802,Widowed,Adm-clerical,Unmarried,White,Female,0.0,0.0,-0.171298,United-States,<=50K
32559,-1.221282,Private,0.154118,HS-grad,-0.441802,Never-married,Adm-clerical,Own-child,White,Male,0.0,0.0,-1.459102,United-States,<=50K


In [8]:
categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# we need this for later 
category_mappings = {}

for col in categorical_columns:
	df[col], uniques = pd.factorize(df[col])
	category_mappings[col] = uniques  
	print(f"{col} convert completed!")

df

workclass convert completed!
education convert completed!
marital-status convert completed!
occupation convert completed!
relationship convert completed!
race convert completed!
sex convert completed!
native-country convert completed!
income convert completed!


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.032782,0,-1.149114,0,1.156144,0,0,0,0,0,0.0,0.0,-0.171298,0,0
1,0.844236,1,-1.088196,0,1.156144,1,1,1,0,0,0.0,0.0,-1.459102,0,0
2,-0.040986,2,0.302927,1,-0.441802,2,2,0,0,0,0.0,0.0,-0.171298,0,0
3,1.065541,2,0.503446,2,-1.240775,1,2,1,1,0,0.0,0.0,-0.171298,0,0
4,-0.778671,2,1.593428,0,1.156144,1,3,2,1,1,0.0,0.0,-0.171298,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,-0.852440,2,0.740821,6,0.756658,1,10,2,0,1,0.0,0.0,-0.493249,0,0
32557,0.106551,2,-0.341172,1,-0.441802,1,9,1,0,0,0.0,0.0,-0.171298,0,1
32558,1.434384,2,-0.367074,1,-0.441802,6,0,4,0,1,0.0,0.0,-0.171298,0,0
32559,-1.221282,2,0.154118,1,-0.441802,0,0,3,0,0,0.0,0.0,-1.459102,0,0


In [9]:
df = df.dropna().reset_index(drop=True)

In [10]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.032782,0,-1.149114,0,1.156144,0,0,0,0,0,0.0,0.0,-0.171298,0,0
1,0.844236,1,-1.088196,0,1.156144,1,1,1,0,0,0.0,0.0,-1.459102,0,0
2,-0.040986,2,0.302927,1,-0.441802,2,2,0,0,0,0.0,0.0,-0.171298,0,0
3,1.065541,2,0.503446,2,-1.240775,1,2,1,1,0,0.0,0.0,-0.171298,0,0
4,-0.778671,2,1.593428,0,1.156144,1,3,2,1,1,0.0,0.0,-0.171298,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,-0.852440,2,0.740821,6,0.756658,1,10,2,0,1,0.0,0.0,-0.493249,0,0
32557,0.106551,2,-0.341172,1,-0.441802,1,9,1,0,0,0.0,0.0,-0.171298,0,1
32558,1.434384,2,-0.367074,1,-0.441802,6,0,4,0,1,0.0,0.0,-0.171298,0,0
32559,-1.221282,2,0.154118,1,-0.441802,0,0,3,0,0,0.0,0.0,-1.459102,0,0


In [11]:
sample_df = df.sample(n=512, random_state=42)  
sample_indices = sample_df.index.to_numpy()     
sample_values  = sample_df.values   
random_row = sample_values.tolist()


In [12]:
df_remaining = df.drop(sample_indices)

In [13]:
all_ref = df_remaining.values.tolist()

In [14]:
def batch_cosine_similarity(query_vectors, reference_vectors, top_k=5):
	
	device = 'cuda' if torch.cuda.is_available() else 'cpu'
	
	if not isinstance(query_vectors, torch.Tensor):
		query_tensor = torch.tensor(query_vectors, dtype=torch.float32, device=device)
	else:
		query_tensor = query_vectors.to(device)
	
	if not isinstance(reference_vectors, torch.Tensor):
		ref_tensor = torch.tensor(reference_vectors, dtype=torch.float32, device=device)
	else:
		ref_tensor = reference_vectors.to(device)
	
	N, D = query_tensor.shape
	M = ref_tensor.shape[0]
	
	print(f"Query shape: {query_tensor.shape}")
	print(f"Reference shape: {ref_tensor.shape}")
	
	all_top_indices = []
	all_top_scores = []
	
	for i in range(N):
		query_single = query_tensor[i:i+1]  # (1, D)
		similarities = F.cosine_similarity(ref_tensor, query_single, dim=1)  # (M,)
		
		top_scores, top_indices = torch.topk(similarities, k=min(top_k, M), largest=True)
		
		all_top_indices.append(top_indices)
		all_top_scores.append(top_scores)
	
	batch_top_indices = torch.stack(all_top_indices)  # (N, k)
	batch_top_scores = torch.stack(all_top_scores)    # (N, k)
	
	return batch_top_indices, batch_top_scores

def batch_cosine_similarity_vectorized(query_vectors, reference_vectors, top_k=5):
	"""
	batched version
	"""
	
	device = 'cuda' if torch.cuda.is_available() else 'cpu'
	
	# Tensor conversion
	if not isinstance(query_vectors, torch.Tensor):
		query_tensor = torch.tensor(query_vectors, dtype=torch.float32, device=device)
	else:
		query_tensor = query_vectors.to(device)
	
	if not isinstance(reference_vectors, torch.Tensor):
		ref_tensor = torch.tensor(reference_vectors, dtype=torch.float32, device=device)
	else:
		ref_tensor = reference_vectors.to(device)
	
	# L2 normalization
	query_norm = F.normalize(query_tensor, p=2, dim=1)    # (N, D)
	ref_norm = F.normalize(ref_tensor, p=2, dim=1)        # (M, D)
	
	similarity_matrix = torch.mm(query_norm, ref_norm.T)  # (N, M)
	
	# Top-k 
	top_scores, top_indices = torch.topk(similarity_matrix, k=min(top_k, ref_tensor.size(0)), 
										dim=1, largest=True)
	
	return top_indices, top_scores


In [15]:
K_value = 15

In [16]:
%%time
top_indices, top_scores = batch_cosine_similarity_vectorized(random_row, all_ref, top_k=K_value)

CPU times: user 121 ms, sys: 258 ms, total: 379 ms
Wall time: 1.39 s


In [17]:
%%time
device = 'cuda'
torch_control = torch.tensor(all_ref, dtype=torch.float32, device=device)
torch_treated = torch.tensor(random_row, dtype=torch.float32, device=device)

values, result = km.batch_topk_optimized_match(
    torch_treated.contiguous(), torch_control.contiguous(), K_value)


CPU times: user 72.7 ms, sys: 140 ms, total: 213 ms
Wall time: 967 ms


In [18]:
cosine_indices = top_indices
klastro_indices = result


In [19]:

def center_gram(G):
    """Centering the Gram matrix."""
    n = G.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ G @ H

def linear_CKA(X, Y):
    """
    Computes linear CKA between two matrices.
    
    X: np.ndarray, shape (n_samples, d1)
    Y: np.ndarray, shape (n_samples, d2)
    
    Returns:
        scalar value in [0, 1] representing similarity.
    """
    X = np.asarray(X)
    Y = np.asarray(Y)

    K = X @ X.T
    L = Y @ Y.T

    K_centered = center_gram(K)
    L_centered = center_gram(L)

    hsic = np.sum(K_centered * L_centered)
    normalization = np.sqrt(np.sum(K_centered ** 2) * np.sum(L_centered ** 2))
    
    return hsic / (normalization + 1e-8)  # small epsilon to avoid div-by-zero


def validate_cka(random_row, cosine_indices, klastro_indices, all_ref, device=None):
    """
    random_row      : (B, d) list / np.ndarray / torch.Tensor
    all_ref         : (M, d) list / np.ndarray / torch.Tensor
    cosine_indices  : (B, k) list / np.ndarray / torch.LongTensor
    klastro_indices : (B, k) list / np.ndarray / torch.LongTensor
    """

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def to_tensor(arr, dtype=torch.float32):
        if isinstance(arr, torch.Tensor):
            return arr.to(device)
        return torch.tensor(arr, dtype=dtype, device=device)

    random_row     = to_tensor(random_row, torch.float32)
    all_ref        = to_tensor(all_ref,    torch.float32)
    cosine_indices = to_tensor(cosine_indices, torch.long)
    klastro_indices= to_tensor(klastro_indices, torch.long)

    # ── (B, d) treated array
    T = random_row.float().cpu().numpy()

  # ── mean control matrix found by each method (B, d)
    C_cos = all_ref[cosine_indices].mean(dim=1).cpu().numpy()
    C_kla = all_ref[klastro_indices].mean(dim=1).cpu().numpy()

    # ── CKA calculation
    cka_cos = linear_CKA(T, C_cos)
    cka_kla = linear_CKA(T, C_kla)

    print(f"\n🔵  Cosine-match   CKA : {cka_cos:.6f}")
    print(f"🔴  KlastroKnowledge    CKA : {cka_kla:.6f}")

    winner = "KlastroKnowledge" if cka_kla > cka_cos else "Cosine"
    print(f"\n🏆  Higher CKA → {winner}")

    return {
        "cka_cosine":  cka_cos,
        "cka_klastro": cka_kla,
        "winner":      winner
    }

In [20]:
cosine_indices = top_indices 
klastro_indices = result
results = validate_cka(random_row, cosine_indices, klastro_indices, all_ref)


🔵  Cosine-match   CKA : 0.927594
🔴  KlastroKnowledge    CKA : 0.938028

🏆  Higher CKA → KlastroKnowledge


In [21]:
match_tensor_cosine = top_indices.cpu().numpy()
match_tensor_klastroKnowledge = result.cpu().numpy()


In [22]:
treated_index = 10
control_indices_cosine = match_tensor_cosine[treated_index]

control_indices_klastroKnowledge = match_tensor_klastroKnowledge[treated_index]


In [23]:
sample_arr = np.array(random_row)                      # (100, d)
sample_df  = pd.DataFrame(sample_arr,
						 index   = sample_indices,     # recover original index
						 columns = df.columns)         # 


In [24]:
sample_df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
14160,-0.852440,2.0,-0.280160,5.0,-0.042315,2.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.493249,0.0,0.0
27048,0.475393,0.0,-1.432405,1.0,-0.441802,1.0,1.0,2.0,0.0,1.0,0.0,0.0,-0.171298,0.0,0.0
28868,-0.704903,2.0,-0.009683,0.0,1.156144,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.760407,0.0,1.0
5667,-0.631134,2.0,0.033754,0.0,1.156144,0.0,9.0,0.0,0.0,1.0,0.0,0.0,-0.171298,0.0,0.0
7827,-0.704903,1.0,0.026458,5.0,-0.042315,2.0,6.0,0.0,0.0,0.0,0.0,0.0,1.438456,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15782,-1.516356,5.0,-0.165472,5.0,-0.042315,0.0,11.0,3.0,0.0,1.0,0.0,0.0,-0.171298,0.0,0.0
4280,-0.631134,2.0,2.407888,7.0,0.357171,1.0,2.0,1.0,0.0,0.0,0.0,0.0,-0.171298,5.0,0.0
18089,-1.368819,2.0,0.339468,5.0,-0.042315,0.0,4.0,0.0,0.0,0.0,0.0,0.0,-0.171298,0.0,0.0
5836,0.180319,2.0,-0.978008,3.0,1.555631,1.0,3.0,1.0,0.0,0.0,0.0,0.0,-0.493249,0.0,1.0


In [25]:
trace_cos, trace_kla = [], []

control_idx_array = df.index.to_numpy()  
for i in range(len(sample_df)):              # == 100
	treated_row  = sample_df.iloc[[i]]       # (1, d) DataFrame
 
	cos_ctrl = df.loc[match_tensor_cosine[i]]
	kla_ctrl = df.loc[match_tensor_klastroKnowledge[i]]

	# 
	df_cos_stack = pd.concat([treated_row.assign(group='treated'),
							  cos_ctrl.assign(group='cosine')],
							  ignore_index=True)

	df_kla_stack = pd.concat([treated_row.assign(group='treated'),
							  kla_ctrl.assign(group='klastroknowledge')],
							  ignore_index=True)

	# (4) compute covariance matrix (rowvar=False → columns=features)
	cov_cos = np.cov(df_cos_stack.drop(columns='group').to_numpy(), rowvar=False)
	cov_kla = np.cov(df_kla_stack.drop(columns='group').to_numpy(), rowvar=False)

	# (5) store trace (sum of variances)
	trace_cos.append(np.trace(cov_cos))
	trace_kla.append(np.trace(cov_kla))

In [26]:
trace_cos  = np.array(trace_cos)
trace_kla  = np.array(trace_kla)

print(f"Mean trace – Cosine : {trace_cos.mean():.6f}")
print(f"Mean trace – KlastroKnowledge: {trace_kla.mean():.6f}")

improve_pct = (trace_cos.mean() - trace_kla.mean()) / trace_cos.mean() * 100

print(f"Average reduction: {improve_pct:.2f}%")  


Mean trace – Cosine : 58.747083
Mean trace – KlastroKnowledge: 56.747304
Average reduction: 3.40%


In [27]:
# trace_cos, trace_kla  ←  # 1-D np.array of length B computed above
eps = 1e-8                             # # small constant to prevent log(0)
log_cos = np.log(trace_cos + eps)
log_kla = np.log(trace_kla + eps)

# 1. mean of log-space differences → mean of log ratios
mean_log_diff = (log_cos - log_kla).mean()          # Scalar
#    = mean[log(trace_cos / trace_kla)]

# 2. exponentiate back → geometric mean ratio
geo_ratio = np.exp(mean_log_diff)                   # >1 → Cosine has larger variance
reduction_pct = (1 - 1/geo_ratio) * 100             

print(f"Geometric mean ratio (Cosine / KlastroKnowledge) : {geo_ratio:.4f}")
print(f"⇒ KlastroKnowledge Covariance is on average {reduction_pct:.2f}% lower than Cosine ")

Geometric mean ratio (Cosine / KlastroKnowledge) : 1.0326
⇒ KlastroKnowledge Covariance is on average 3.16% lower than Cosine 


### Even in this low-dimensional tabular setting where cosine already performs well, a CKA gap of ~0.01 consistently in favor of KlastroKnowledge suggests meaningful improvement in representation alignment — and the advantage is expected to widen significantly in high-dimensional RAG retrieval scenarios, where K >= 15